# Analyse 3D des Matrices de Dose — Cohorte FCCSS

**Projet** : Pôle Projet P15 (CentraleSupélec)  
**Objectif** : Visualisation et analyse des matrices de dose 3D cardiaques issues du répertoire `coeur_1k`.  
**Données** : Fichiers `dosi_coeur_<ctr>_<numcent>.csv` (coordonnées x, y, z + dose en Gy)

---

## Contenu

1. **Configuration et chargement** — Sélection d'un patient avec Pathologie_cardiaque == 1
2. **Visualisation 3D statique** — Matplotlib scatter 3D coloré par dose
3. **Visualisation 3D interactive** — Plotly (isosurfaces et volume rendering)
4. **Analyse DVH** — Histogramme dose-volume
5. **Dashboard clinique** — Vue synthétique multi-panels
6. **Alternatives et perspectives** — Comparaison PyVista, Mayavi

---
## 0. Configuration et imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
from scipy import stats
from pathlib import Path
import os

# Plotly for interactive 3D
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    print(" Plotly not installed. Install with: pip install plotly")
    PLOTLY_AVAILABLE = False

# ------------------------------------------------------------------
# Constants
# ------------------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = Path('../../data/RT_Thorax_v1.csv')
COEUR_PATH = Path('../../data/coeur_1k/')
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Publication style
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'Georgia'],
    'font.size': 11,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.titleweight': 'bold',
})

# Project color palette
COLORS = {
    'blue': '#1f4e79',
    'green': '#2e8b57',
    'orange': '#c17c32',
    'purple': '#6a4c93',
    'red': '#c0392b',
}

# Clinical dose thresholds (Gy) for isosurfaces
DOSE_THRESHOLDS = [5, 10, 15, 20]

print(" Configuration loaded")

---
## 1. Chargement des données et sélection du patient

In [ ]:
# Load clinical dataset
df = pd.read_csv(DATA_PATH)
print(f"Patients total: {len(df)}")

# Filter patients with cardiac pathology
patients_pos = df[df['Pathologie_cardiaque'] == 1].copy()
print(f"Patients avec pathologie cardiaque: {len(patients_pos)}")

In [ ]:
def find_patient_with_dose_file(patients_df, coeur_path):
    """Find a patient with Pathologie_cardiaque==1 who has a dose file."""
    # Method 1: Direct lookup by ctr/numcent
    for _, row in patients_df.iterrows():
        ctr = int(row['ctr'])
        numcent = int(row['numcent'])
        filename = f"dosi_coeur_{ctr}_{numcent}.csv"
        filepath = coeur_path / filename
        if filepath.exists():
            return row, filepath
    
    # Method 2: Scan directory and match
    print("  Direct lookup failed, scanning directory...")
    if not coeur_path.exists():
        raise FileNotFoundError(f"Directory not found: {coeur_path}")
    
    for f in sorted(coeur_path.iterdir()):
        if not f.suffix == '.csv':
            continue
        parts = f.stem.split('_')
        if len(parts) >= 4:
            try:
                ctr_f, numcent_f = int(parts[2]), int(parts[3])
            except ValueError:
                continue
            match = patients_df[
                (patients_df['ctr'] == ctr_f) & 
                (patients_df['numcent'] == numcent_f)
            ]
            if len(match) > 0:
                print(f"  Found via file scan: {f.name}")
                return match.iloc[0], f
    
    return None, None

# Find patient
selected_patient, dose_file = find_patient_with_dose_file(patients_pos, COEUR_PATH)

if selected_patient is None:
    print(" Aucun patient avec fichier de dose trouvé.")
    print("   Vérifiez que le répertoire data/coeur_1k/ contient les fichiers.")
else:
    print(f"\n Patient sélectionné:")
    print(f"  CTR: {int(selected_patient['ctr'])}, NUMCENT: {int(selected_patient['numcent'])}")
    print(f"  Dose moyenne: {selected_patient['mean']:.2f} Gy")
    print(f"  Âge au diagnostic: {selected_patient['age_diag']:.0f} ans")
    print(f"  Type ICCC: {selected_patient['iccc_type']}")
    print(f"  Fichier: {dose_file.name}")

In [ ]:
def load_dose_matrix(filepath):
    """Load dose matrix from CSV file."""
    # Try tab-separated first (common format)
    try:
        dose_df = pd.read_csv(filepath, sep='\t')
    except:
        dose_df = pd.read_csv(filepath)
    
    print(f"Shape: {dose_df.shape}")
    print(f"Columns: {list(dose_df.columns)}")
    
    # Identify coordinate columns
    x_col = [c for c in dose_df.columns if c.strip().lower() == 'x']
    y_col = [c for c in dose_df.columns if c.strip().lower() == 'y']
    z_col = [c for c in dose_df.columns if c.strip().lower() == 'z']
    
    if x_col and y_col and z_col:
        x_col, y_col, z_col = x_col[0], y_col[0], z_col[0]
    else:
        # Fallback to positional
        x_col = dose_df.columns[3]
        y_col = dose_df.columns[4]
        z_col = dose_df.columns[5]
    
    # Dose column: last numeric column
    dose_col = dose_df.columns[-1]
    print(f"Using: x={x_col}, y={y_col}, z={z_col}, dose={dose_col}")
    
    # Clean and extract
    data = dose_df[[x_col, y_col, z_col, dose_col]].copy()
    data.columns = ['x', 'y', 'z', 'dose']
    data = data.dropna(subset=['dose'])
    data['dose'] = pd.to_numeric(data['dose'], errors='coerce')
    data = data.dropna()
    
    return data

# Load dose data
if dose_file is not None:
    dose_data = load_dose_matrix(dose_file)
    print(f"\n Voxels valides: {len(dose_data)}")
    print(f"  Dose range: [{dose_data['dose'].min():.2f}, {dose_data['dose'].max():.2f}] Gy")
    print(f"  Dose moyenne: {dose_data['dose'].mean():.2f} Gy")
    print(f"  Dose médiane: {dose_data['dose'].median():.2f} Gy")
else:
    dose_data = None

---
## 2. Visualisation 3D statique (Matplotlib)

In [ ]:
def plot_3d_static(dose_data, max_points=30000):
    """Static 3D scatter plot colored by dose."""
    if dose_data is None:
        print("No dose data available.")
        return None
    
    # Subsample if too many points
    if len(dose_data) > max_points:
        plot_data = dose_data.sample(n=max_points, random_state=RANDOM_STATE)
        print(f"Subsampled to {max_points} voxels for visualization")
    else:
        plot_data = dose_data
    
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Color by dose
    norm = Normalize(vmin=0, vmax=plot_data['dose'].quantile(0.95))
    colors = cm.rainbow(norm(plot_data['dose']))
    
    scatter = ax.scatter(
        plot_data['x'], plot_data['y'], plot_data['z'],
        c=plot_data['dose'], cmap='rainbow',
        s=1, alpha=0.6,
        vmin=0, vmax=plot_data['dose'].quantile(0.95)
    )
    
    # Colorbar
    cbar = plt.colorbar(scatter, ax=ax, shrink=0.6, pad=0.1)
    cbar.set_label('Dose (Gy)', fontsize=12)
    
    ax.set_xlabel('X (mm)', fontsize=11)
    ax.set_ylabel('Y (mm)', fontsize=11)
    ax.set_zlabel('Z (mm)', fontsize=11)
    ax.set_title('Distribution 3D de la dose cardiaque\n(Patient avec pathologie)', fontsize=13, fontweight='bold')
    
    # Add patient info
    info_text = (
        f"Dose moyenne: {dose_data['dose'].mean():.1f} Gy\n"
        f"Dose max: {dose_data['dose'].max():.1f} Gy\n"
        f"Voxels: {len(dose_data):,}"
    )
    ax.text2D(0.02, 0.98, info_text, transform=ax.transAxes, fontsize=10,
              verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    return fig

if dose_data is not None:
    fig_static = plot_3d_static(dose_data)
    fig_static.savefig(FIG_DIR / 'dose_3d_static.png', dpi=300, bbox_inches='tight')
    print(f" Saved {FIG_DIR / 'dose_3d_static.png'}")
    plt.show()

---
## 3. Visualisation 3D interactive (Plotly)

### 3.1 Scatter 3D interactif

In [ ]:
def plot_3d_interactive_scatter(dose_data, max_points=50000):
    """Interactive 3D scatter plot with Plotly."""
    if not PLOTLY_AVAILABLE or dose_data is None:
        print("Plotly not available or no data.")
        return None
    
    # Subsample
    if len(dose_data) > max_points:
        plot_data = dose_data.sample(n=max_points, random_state=RANDOM_STATE)
    else:
        plot_data = dose_data
    
    fig = go.Figure(data=[go.Scatter3d(
        x=plot_data['x'],
        y=plot_data['y'],
        z=plot_data['z'],
        mode='markers',
        marker=dict(
            size=2,
            color=plot_data['dose'],
            colorscale='rainbow',
            colorbar=dict(title='Dose (Gy)'),
            opacity=0.7,
            cmin=0,
            cmax=plot_data['dose'].quantile(0.95)
        ),
        hovertemplate='X: %{x:.1f}<br>Y: %{y:.1f}<br>Z: %{z:.1f}<br>Dose: %{marker.color:.2f} Gy<extra></extra>'
    )])
    
    fig.update_layout(
        title=dict(
            text='Distribution 3D de la dose cardiaque (Patient avec pathologie)',
            font=dict(size=16)
        ),
        scene=dict(
            xaxis_title='X (mm)',
            yaxis_title='Y (mm)',
            zaxis_title='Z (mm)',
            aspectmode='data'
        ),
        width=900,
        height=700
    )
    
    return fig

if dose_data is not None and PLOTLY_AVAILABLE:
    fig_interactive = plot_3d_interactive_scatter(dose_data)
    fig_interactive.write_html(FIG_DIR / 'dose_3d_interactive.html', include_plotlyjs='cdn')
    print(f" Saved {FIG_DIR / 'dose_3d_interactive.html'}")
    fig_interactive.show()

### 3.2 Isosurfaces à seuils cliniques

In [ ]:
def plot_isosurfaces(dose_data, thresholds=[5, 10, 15, 20], grid_resolution=50):
    """
    Create isosurface visualization at clinical dose thresholds.
    Interpolates scattered points to a regular grid.
    """
    if not PLOTLY_AVAILABLE or dose_data is None:
        return None
    
    print(f"Creating isosurfaces at thresholds: {thresholds} Gy")
    
    # Create regular grid
    x_range = np.linspace(dose_data['x'].min(), dose_data['x'].max(), grid_resolution)
    y_range = np.linspace(dose_data['y'].min(), dose_data['y'].max(), grid_resolution)
    z_range = np.linspace(dose_data['z'].min(), dose_data['z'].max(), grid_resolution)
    
    X, Y, Z = np.meshgrid(x_range, y_range, z_range, indexing='ij')
    
    # Interpolate dose to grid (this can be slow for large datasets)
    print("  Interpolating to regular grid...")
    points = dose_data[['x', 'y', 'z']].values
    values = dose_data['dose'].values
    
    # Use nearest neighbor for speed (linear is more accurate but slower)
    dose_grid = griddata(points, values, (X, Y, Z), method='nearest', fill_value=0)
    
    # Create isosurface traces
    colorscale = ['rgba(0,100,200,0.3)', 'rgba(0,200,100,0.4)', 
                  'rgba(255,165,0,0.5)', 'rgba(255,0,0,0.6)']
    
    traces = []
    for i, threshold in enumerate(thresholds):
        traces.append(go.Isosurface(
            x=X.flatten(),
            y=Y.flatten(),
            z=Z.flatten(),
            value=dose_grid.flatten(),
            isomin=threshold,
            isomax=threshold,
            surface_count=1,
            colorscale=[[0, colorscale[i]], [1, colorscale[i]]],
            showscale=False,
            name=f'{threshold} Gy',
            opacity=0.5
        ))
    
    fig = go.Figure(data=traces)
    fig.update_layout(
        title='Isosurfaces de dose cardiaque (V5, V10, V15, V20)',
        scene=dict(
            xaxis_title='X (mm)',
            yaxis_title='Y (mm)',
            zaxis_title='Z (mm)',
            aspectmode='data'
        ),
        width=900,
        height=700
    )
    
    return fig

if dose_data is not None and PLOTLY_AVAILABLE and len(dose_data) < 200000:
    fig_iso = plot_isosurfaces(dose_data, DOSE_THRESHOLDS)
    if fig_iso:
        fig_iso.write_html(FIG_DIR / 'dose_3d_isosurface.html', include_plotlyjs='cdn')
        print(f" Saved {FIG_DIR / 'dose_3d_isosurface.html'}")
        fig_iso.show()
else:
    print("Skipping isosurface (dataset too large or missing dependencies)")

### 3.3 Carte d'activation cardiaque — Visualisation réaliste

Reconstruction de la **surface du cœur** à partir du nuage de points (alpha shapes via triangulation de Delaunay), puis projection de la dose comme une **carte d'activation** (similaire aux cartes d'activation électrophysiologiques en cardiologie interventionnelle).

- **Surface** : extraction des faces externes via alpha shapes (filtrage par longueur d'arête maximale)
- **Coloration** : dose projetée sur chaque sommet, colormap clinique bleu → vert → jaune → rouge
- **Éclairage Phong** : ambient + diffuse + spéculaire pour un rendu 3D réaliste
- **Vues multiples** : antérieure, latérale, supérieure

In [ ]:
from scipy.spatial import Delaunay, ConvexHull

def extract_alpha_surface(points, alpha):
    """
    Extraction de la surface externe via alpha shapes.
    Filtre les tetraedres de Delaunay par longueur d'arete maximale,
    puis extrait les faces qui n'appartiennent qu'a un seul tetraedre (= surface).
    """
    tri = Delaunay(points)
    simplices = tri.simplices
    face_count = {}

    for simplex in simplices:
        pts_t = points[simplex]
        # Longueur d'arete maximale du tetraedre
        max_edge = 0
        for i in range(4):
            for j in range(i + 1, 4):
                d = np.linalg.norm(pts_t[i] - pts_t[j])
                if d > max_edge:
                    max_edge = d

        if max_edge < alpha:
            # Ajouter les 4 faces triangulaires du tetraedre
            for i in range(4):
                face = tuple(sorted(simplex[j] for j in range(4) if j != i))
                face_count[face] = face_count.get(face, 0) + 1

    # Faces n'apparaissant qu'une fois = faces de surface
    surface_faces = np.array([f for f, c in face_count.items() if c == 1])
    return surface_faces


def plot_cardiac_activation_map(dose_data, alpha=6.0):
    """
    Carte d'activation cardiaque 3D realiste.
    Reconstruit la surface via alpha shapes et projette la dose.
    """
    if not PLOTLY_AVAILABLE or dose_data is None:
        print("Plotly non disponible ou donnees manquantes.")
        return None

    points = dose_data[['x', 'y', 'z']].values
    dose_vals = dose_data['dose'].values

    # --- Extraction de la surface via alpha shapes ---
    print(f"Extraction de la surface (alpha={alpha})...")
    surface_faces = extract_alpha_surface(points, alpha)
    print(f"  {len(surface_faces)} faces de surface extraites")

    if len(surface_faces) == 0:
        print("Aucune face extraite. Essayez un alpha plus grand.")
        return None

    # --- Colormap clinique type carte d'activation ---
    # Bleu froid (faible dose) -> Vert -> Jaune -> Rouge vif (forte dose)
    activation_colorscale = [
        [0.00, 'rgb(8, 29, 88)'],       # bleu tres fonce
        [0.10, 'rgb(34, 94, 168)'],      # bleu
        [0.25, 'rgb(29, 145, 192)'],     # bleu-vert
        [0.40, 'rgb(65, 182, 196)'],     # cyan
        [0.50, 'rgb(127, 205, 187)'],    # vert clair
        [0.60, 'rgb(199, 233, 180)'],    # vert-jaune
        [0.70, 'rgb(237, 248, 177)'],    # jaune clair
        [0.80, 'rgb(255, 217, 47)'],     # jaune
        [0.90, 'rgb(253, 141, 60)'],     # orange
        [0.95, 'rgb(227, 26, 28)'],      # rouge
        [1.00, 'rgb(128, 0, 38)'],       # rouge fonce
    ]

    i_faces = surface_faces[:, 0]
    j_faces = surface_faces[:, 1]
    k_faces = surface_faces[:, 2]

    dose_95 = np.percentile(dose_vals, 95)

    # --- Figure principale : carte d'activation ---
    fig = go.Figure()

    # Surface cardiaque avec dose
    fig.add_trace(go.Mesh3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        i=i_faces, j=j_faces, k=k_faces,
        intensity=dose_vals,
        intensitymode='vertex',
        colorscale=activation_colorscale,
        cmin=0, cmax=dose_95,
        colorbar=dict(
            title=dict(text='Dose (Gy)', font=dict(size=14)),
            thickness=20, len=0.7,
            tickfont=dict(size=12)
        ),
        lighting=dict(
            ambient=0.3,
            diffuse=0.7,
            specular=0.4,
            roughness=0.3,
            fresnel=0.2
        ),
        lightposition=dict(x=100, y=200, z=300),
        flatshading=False,
        opacity=1.0,
        name='Surface cardiaque',
        hovertemplate=(
            'X: %{x:.1f} mm<br>'
            'Y: %{y:.1f} mm<br>'
            'Z: %{z:.1f} mm<br>'
            'Dose: %{intensity:.2f} Gy'
            '<extra></extra>'
        )
    ))

    # --- Layout avec vue anterieure par defaut ---
    cx = points[:, 0].mean()
    cy = points[:, 1].mean()
    cz = points[:, 2].mean()

    # Boutons pour vues predefinies (anterieure, laterale, superieure)
    camera_views = {
        'Anterieure': dict(eye=dict(x=0, y=-2.0, z=0.3), up=dict(x=0, y=0, z=1)),
        'Posterieure': dict(eye=dict(x=0, y=2.0, z=0.3), up=dict(x=0, y=0, z=1)),
        'Laterale G': dict(eye=dict(x=-2.0, y=0, z=0.3), up=dict(x=0, y=0, z=1)),
        'Laterale D': dict(eye=dict(x=2.0, y=0, z=0.3), up=dict(x=0, y=0, z=1)),
        'Superieure': dict(eye=dict(x=0, y=-0.3, z=2.0), up=dict(x=0, y=-1, z=0)),
        'Inferieure': dict(eye=dict(x=0, y=-0.3, z=-2.0), up=dict(x=0, y=-1, z=0)),
    }

    buttons = []
    for name, cam in camera_views.items():
        buttons.append(dict(
            label=name,
            method='relayout',
            args=[{'scene.camera': cam}]
        ))

    fig.update_layout(
        title=dict(
            text=('Carte d\'activation dosimetrique cardiaque<br>'
                  '<sub>Surface reconstruite par alpha shapes — Dose projetee (Gy)</sub>'),
            font=dict(size=16),
            x=0.5
        ),
        scene=dict(
            xaxis=dict(title='X (mm)', showgrid=True, gridcolor='rgba(0,0,0,0.1)',
                       backgroundcolor='rgb(240,240,240)'),
            yaxis=dict(title='Y (mm)', showgrid=True, gridcolor='rgba(0,0,0,0.1)',
                       backgroundcolor='rgb(240,240,240)'),
            zaxis=dict(title='Z (mm)', showgrid=True, gridcolor='rgba(0,0,0,0.1)',
                       backgroundcolor='rgb(240,240,240)'),
            aspectmode='data',
            camera=camera_views['Anterieure']
        ),
        updatemenus=[
            dict(
                type='buttons',
                showactive=True,
                buttons=buttons,
                x=0.02, y=0.98,
                xanchor='left', yanchor='top',
                direction='down',
                bgcolor='rgba(255,255,255,0.8)',
                bordercolor='rgba(0,0,0,0.3)',
                font=dict(size=11)
            )
        ],
        width=950,
        height=750,
        paper_bgcolor='white',
    )

    return fig


# --- Execution ---
if dose_data is not None and PLOTLY_AVAILABLE:
    fig_activation = plot_cardiac_activation_map(dose_data, alpha=6.0)
    if fig_activation:
        fig_activation.write_html(
            FIG_DIR / 'dose_3d_activation_map.html',
            include_plotlyjs='cdn'
        )
        print(f"\u2713 Saved {FIG_DIR / 'dose_3d_activation_map.html'}")
        fig_activation.show()
else:
    print("Donnees ou Plotly non disponibles.")

---
## 4. Analyse DVH (Dose-Volume Histogram)

In [ ]:
def compute_dvh(dose_data, bins=100):
    """Compute cumulative Dose-Volume Histogram."""
    if dose_data is None:
        return None, None
    
    doses = dose_data['dose'].values
    dose_bins = np.linspace(0, doses.max(), bins)
    
    # Cumulative DVH: percentage of volume receiving >= dose
    dvh = np.array([100 * np.sum(doses >= d) / len(doses) for d in dose_bins])
    
    return dose_bins, dvh

def plot_dvh(dose_data):
    """Plot DVH with clinical thresholds."""
    if dose_data is None:
        print("No dose data available.")
        return None
    
    dose_bins, dvh = compute_dvh(dose_data)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot DVH curve
    ax.plot(dose_bins, dvh, color=COLORS['blue'], linewidth=2, label='DVH cumulatif')
    ax.fill_between(dose_bins, dvh, alpha=0.3, color=COLORS['blue'])
    
    # Add clinical thresholds
    for d in DOSE_THRESHOLDS:
        if d <= dose_bins.max():
            v = np.interp(d, dose_bins, dvh)
            ax.axvline(d, color=COLORS['orange'], linestyle='--', alpha=0.7)
            ax.scatter([d], [v], color=COLORS['red'], s=50, zorder=5)
            ax.annotate(f'V{d}={v:.1f}%', xy=(d, v), xytext=(d+1, v+5),
                       fontsize=10, color=COLORS['red'])
    
    ax.set_xlabel('Dose (Gy)', fontsize=12)
    ax.set_ylabel('Volume (%)', fontsize=12)
    ax.set_title('Histogramme Dose-Volume (DVH) — Cœur', fontsize=13, fontweight='bold')
    ax.set_xlim(0, dose_bins.max())
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    # Add statistics
    stats_text = (
        f"Dose moyenne: {dose_data['dose'].mean():.1f} Gy\n"
        f"Dose médiane: {dose_data['dose'].median():.1f} Gy\n"
        f"Dose max: {dose_data['dose'].max():.1f} Gy"
    )
    ax.text(0.98, 0.5, stats_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='center', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    return fig

if dose_data is not None:
    fig_dvh = plot_dvh(dose_data)
    fig_dvh.savefig(FIG_DIR / 'dose_dvh.png', dpi=300, bbox_inches='tight')
    print(f" Saved {FIG_DIR / 'dose_dvh.png'}")
    plt.show()

---
## 5. Dashboard clinique multi-panels

In [ ]:
def plot_clinical_dashboard(dose_data, patient_info=None):
    """Multi-panel dashboard: 3 slices + DVH + histogram."""
    if dose_data is None:
        print("No dose data available.")
        return None
    
    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)
    
    # Get dose range for consistent colormap
    vmax = dose_data['dose'].quantile(0.95)
    
    # Panel 1-3: Axial slices at different Z levels
    z_values = np.percentile(dose_data['z'], [25, 50, 75])
    slice_titles = ['Coupe inférieure (Z25%)', 'Coupe médiane (Z50%)', 'Coupe supérieure (Z75%)']
    
    for i, (z_val, title) in enumerate(zip(z_values, slice_titles)):
        ax = fig.add_subplot(gs[0, i])
        
        # Select voxels near this Z level
        z_tol = (dose_data['z'].max() - dose_data['z'].min()) / 20
        slice_data = dose_data[np.abs(dose_data['z'] - z_val) < z_tol]
        
        if len(slice_data) > 0:
            scatter = ax.scatter(
                slice_data['x'], slice_data['y'],
                c=slice_data['dose'], cmap='hot',
                s=5, alpha=0.8, vmin=0, vmax=vmax
            )
            plt.colorbar(scatter, ax=ax, label='Dose (Gy)')
        
        ax.set_xlabel('X (mm)')
        ax.set_ylabel('Y (mm)')
        ax.set_title(title, fontweight='bold')
        ax.set_aspect('equal')
    
    # Panel 4: DVH
    ax_dvh = fig.add_subplot(gs[1, 0])
    dose_bins, dvh = compute_dvh(dose_data)
    ax_dvh.plot(dose_bins, dvh, color=COLORS['blue'], linewidth=2)
    ax_dvh.fill_between(dose_bins, dvh, alpha=0.3, color=COLORS['blue'])
    for d in DOSE_THRESHOLDS:
        if d <= dose_bins.max():
            ax_dvh.axvline(d, color=COLORS['orange'], linestyle='--', alpha=0.5)
    ax_dvh.set_xlabel('Dose (Gy)')
    ax_dvh.set_ylabel('Volume (%)')
    ax_dvh.set_title('DVH cumulatif', fontweight='bold')
    ax_dvh.grid(True, alpha=0.3)
    
    # Panel 5: Dose histogram
    ax_hist = fig.add_subplot(gs[1, 1])
    ax_hist.hist(dose_data['dose'], bins=50, color=COLORS['green'], alpha=0.7, edgecolor='white')
    ax_hist.axvline(dose_data['dose'].mean(), color=COLORS['red'], linestyle='--', 
                    linewidth=2, label=f"Moyenne: {dose_data['dose'].mean():.1f} Gy")
    ax_hist.axvline(dose_data['dose'].median(), color=COLORS['purple'], linestyle=':', 
                    linewidth=2, label=f"Médiane: {dose_data['dose'].median():.1f} Gy")
    ax_hist.set_xlabel('Dose (Gy)')
    ax_hist.set_ylabel('Nombre de voxels')
    ax_hist.set_title('Distribution des doses', fontweight='bold')
    ax_hist.legend(fontsize=9)
    
    # Panel 6: Statistics
    ax_stats = fig.add_subplot(gs[1, 2])
    ax_stats.axis('off')
    
    # Compute Vx values
    vx_values = {}
    for d in DOSE_THRESHOLDS:
        vx_values[f'V{d}'] = 100 * np.sum(dose_data['dose'] >= d) / len(dose_data)
    
    stats_text = (
        f"STATISTIQUES DOSIMÉTRIQUES\n"
        f"{'='*30}\n\n"
        f"Voxels totaux: {len(dose_data):,}\n\n"
        f"Dose moyenne: {dose_data['dose'].mean():.2f} Gy\n"
        f"Dose médiane: {dose_data['dose'].median():.2f} Gy\n"
        f"Dose max: {dose_data['dose'].max():.2f} Gy\n"
        f"Écart-type: {dose_data['dose'].std():.2f} Gy\n\n"
        f"{'='*30}\n\n"
        f"V5: {vx_values['V5']:.1f}%\n"
        f"V10: {vx_values['V10']:.1f}%\n"
        f"V15: {vx_values['V15']:.1f}%\n"
        f"V20: {vx_values['V20']:.1f}%"
    )
    
    ax_stats.text(0.1, 0.95, stats_text, transform=ax_stats.transAxes,
                  fontsize=11, verticalalignment='top', fontfamily='monospace',
                  bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.8))
    
    fig.suptitle('Dashboard Dosimétrique — Cœur (Patient avec pathologie cardiaque)',
                 fontsize=14, fontweight='bold', y=0.98)
    
    return fig

if dose_data is not None:
    fig_dashboard = plot_clinical_dashboard(dose_data)
    fig_dashboard.savefig(FIG_DIR / 'dose_3d_clinical_dashboard.png', dpi=300, bbox_inches='tight')
    print(f" Saved {FIG_DIR / 'dose_3d_clinical_dashboard.png'}")
    plt.show()

---
## 6. Alternatives et perspectives

### Comparaison des bibliothèques de visualisation 3D

| Bibliothèque | Avantages | Inconvénients | Cas d'usage |
|--------------|-----------|---------------|-------------|
| **Plotly** | Interactif, export HTML, pas d'install GPU | Fichiers lourds, lent sur >100k points | Exploration rapide, partage web |
| **PyVista** | Rendu VTK haute qualité, isosurfaces | Nécessite VTK, moins portable | Publication, isosurfaces précises |
| **Mayavi** | Rendu scientifique professionnel | Installation complexe, dépendances lourdes | Visualisation avancée recherche |
| **Matplotlib 3D** | Simple, intégré, statique | Qualité limitée, pas d'interaction | Figures rapides, baseline |

### Prochaines étapes (Phase 3)

1. **Recalage des volumes** — ANTsPy pour aligner les cœurs sur un atlas commun
2. **Features dosiomiques** — Extraction de texture (GLCM, GLRLM) et forme
3. **Cartes d'activation** — Régression voxel-wise pour identifier les zones à risque

In [ ]:
# Example: PyVista alternative (requires: pip install pyvista)
# Uncomment to use

# try:
#     import pyvista as pv
#     
#     if dose_data is not None:
#         # Create point cloud
#         points = dose_data[['x', 'y', 'z']].values
#         cloud = pv.PolyData(points)
#         cloud['dose'] = dose_data['dose'].values
#         
#         # Plot
#         plotter = pv.Plotter()
#         plotter.add_mesh(cloud, scalars='dose', cmap='hot', 
#                          point_size=3, render_points_as_spheres=True)
#         plotter.add_scalar_bar('Dose (Gy)')
#         plotter.show()
# except ImportError:
#     print("PyVista not installed. pip install pyvista")

---
## Résumé des sorties

| Fichier | Description |
|---------|-------------|
| `figures/dose_3d_static.png` | Visualisation 3D statique (matplotlib) |
| `figures/dose_3d_interactive.html` | Scatter 3D interactif (plotly) |
| `figures/dose_3d_isosurface.html` | Isosurfaces V5/V10/V15/V20 (plotly) |
| `figures/dose_dvh.png` | Histogramme dose-volume |
| `figures/dose_3d_clinical_dashboard.png` | Dashboard multi-panels |